# Predicting Heart Disease
The World Health Organization (WHO) estimates that 17.9 million people die every year because of cardiovascular diseases (CVDs).

There are multiple risk factors that could contribute to CVD in an individual such as unhealthy diet, lack of physical activity or mental illnesses. Being able to identify these risk factors in individuals early on could help prevent a lot of premature deaths.

In this project, we will use the [Kaggle dataset](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) and build a K-Nearest Neighbors classifier to accurately predict the likelihood of a patient having a heart disease in the future. 

## EDA: Descriptive Statistics

We will start with exploring our dataset. As per the source, each patient has the following information collected about them:


1. `Age`: age of the patient [years]
2. `Sex`: sex of the patient [M: Male, F: Female]
3. `ChestPainType`: chest pain type [TA: Typical Angina, ATA: Atypical Angina, NAP: Non-Anginal Pain, ASY: Asymptomatic]
4. `RestingBP`: resting blood pressure [mm Hg]
5. `Cholesterol`: serum cholesterol [mm/dl]
6. `FastingBS`: fasting blood sugar [1: if FastingBS > 120 mg/dl, 0: otherwise]
7. `RestingECG`: resting electrocardiogram results [Normal: Normal, ST: having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV), LVH: showing probable or definite left ventricular hypertrophy by Estes' criteria]
8. `MaxHR`: maximum heart rate achieved [Numeric value between 60 and 202]
9. `ExerciseAngina`: exercise-induced angina [Y: Yes, N: No]
10. `Oldpeak`: oldpeak = ST [Numeric value measured in depression]
11. `ST_Slope`: the slope of the peak exercise ST segment [Up: upsloping, Flat: flat, Down: downsloping]
12. `HeartDisease`: output class [1: heart disease, 0: Normal]

In [ ]:
import pandas as pd

In [3]:
df = pd.read_csv("heart_disease_prediction.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:
df.shape

(918, 12)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [8]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


From the table above, we can observe that:

- The average age of patients is ~`53` years.
- The median for `Cholesterol` is higher than its mean by roughly `25` mm/dl, indicating that it could be a left-skewed distribution with a possibility of outliers skewing the distribution.
- `RestingBP` and `Cholesterol` have a minimum value of zero.
- There don't seem to be any missing values in these columns. But we will have to confirm it across the entire dataset as well.

`RestingBP` can't be `0`. And, as per the [American Heart Association](https://www.heart.org/en/health-topics/cholesterol/about-cholesterol/what-your-cholesterol-levels-mean), serum cholesterol is a composite of different measurements. So, it is unlikely that `Cholesterol` would be `0` as well. We will have to clean both of these up later.

Next, we will look at the categorical variables. It would also be beneficial to look at how the target feature, `HeartDisease`, is related to those categories. Before that, let's quickly check if there are any missing values in the dataset or not.

In [10]:
df.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

## 🩺 Exploratory Data Analysis: Descriptive Statistics

This dataset contains clinical and demographic information about patients, collected to study the presence of **heart disease**. Below is a detailed description of each feature:

1. **Age**  
   - Patient’s age in years.  
   - Important for risk stratification since cardiovascular risk increases with age.

2. **Sex**  
   - Biological sex of the patient: **M = Male**, **F = Female**.  
   - Heart disease prevalence often differs between men and women.

3. **ChestPainType**  
   - Type of chest pain experienced:  
     - **TA (Typical Angina)**: Classic chest pain related to exertion.  
     - **ATA (Atypical Angina)**: Chest discomfort not following the typical pattern.  
     - **NAP (Non-Anginal Pain)**: Pain not related to heart disease.  
     - **ASY (Asymptomatic)**: No chest pain, but disease may still be present.

4. **RestingBP**  
   - Resting blood pressure in **mm Hg**.  
   - Elevated blood pressure is a major risk factor for heart disease.

5. **Cholesterol**  
   - Serum cholesterol level in **mg/dl**.  
   - High cholesterol is linked to plaque buildup in arteries.

6. **FastingBS**  
   - Fasting blood sugar indicator:  
     - **1** → FastingBS > 120 mg/dl (high).  
     - **0** → Normal fasting blood sugar.  
   - Diabetes is a strong risk factor for cardiovascular disease.

7. **RestingECG**  
   - Resting electrocardiogram results:  
     - **Normal** → No abnormalities.  
     - **ST** → ST-T wave abnormalities (e.g., inversions, elevation, depression).  
     - **LVH** → Left Ventricular Hypertrophy by Estes’ criteria.  
   - ECG abnormalities can indicate underlying heart conditions.

8. **MaxHR**  
   - Maximum heart rate achieved during exercise (range: 60–202 bpm).  
   - Lower maximum heart rate may indicate reduced cardiac fitness.

9. **ExerciseAngina**  
   - Exercise-induced angina: **Y = Yes**, **N = No**.  
   - Presence of angina during exercise is a strong predictor of heart disease.

10. **Oldpeak**  
    - ST depression induced by exercise relative to rest.  
    - Measured as a numeric value; higher values suggest ischemia.

11. **ST_Slope**  
    - Slope of the peak exercise ST segment:  
      - **Up** → Upsloping (generally normal).  
      - **Flat** → Flat slope (possible abnormality).  
      - **Down** → Downsloping (strongly abnormal, linked to ischemia).

12. **HeartDisease**  
    - Target variable:  
      - **1** → Patient diagnosed with heart disease.  
      - **0** → Patient considered normal.
